In [1]:
import sys
sys.path.append("../src")

from pyspark.sql import SparkSession, functions as F
import main, m1

spark = (SparkSession.builder
         .master("local[*]")
         .appName("dev-m2")
         .config("spark.driver.memory", "2g")
         .getOrCreate())

trips = main.load_trips(spark, "../databases", [2025], 1.0)        
clean = m1.clean(m1.flag_invalid(trips))                           
zones = main.load_zones(spark, "../databases/taxi_zone_lookup.csv")   

print(clean.count())
zones.show(5)   

c:\Users\User\AppData\Local\Programs\Python\Python314\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


17483677
+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [2]:
pickup_zone = zones.select(F.col("LocationID").alias("PULocationID"),   
                           F.col("Borough").alias("pickup_borough"))      

with_borough = clean.join(pickup_zone, "PULocationID")
with_borough.select("tpep_pickup_datetime", "PULocationID", "pickup_borough").show(5)
print(with_borough.count())

+--------------------+------------+--------------+
|tpep_pickup_datetime|PULocationID|pickup_borough|
+--------------------+------------+--------------+
| 2025-01-01 00:18:38|         229|     Manhattan|
| 2025-01-01 00:32:40|         236|     Manhattan|
| 2025-01-01 00:44:04|         141|     Manhattan|
| 2025-01-01 00:14:27|         244|     Manhattan|
| 2025-01-01 00:21:34|         244|     Manhattan|
+--------------------+------------+--------------+
only showing top 5 rows
17483677


In [3]:
with_cols = (with_borough
    .withColumn("day_of_week", F.dayofweek("tpep_pickup_datetime")) 
    .withColumn("hour_of_day", F.hour("tpep_pickup_datetime"))     
    .withColumn("duration_min",                                       
                (F.unix_timestamp("tpep_dropoff_datetime") -
                 F.unix_timestamp("tpep_pickup_datetime")) / 60.0))

with_cols.select("tpep_pickup_datetime", "tpep_dropoff_datetime",
                 "day_of_week", "hour_of_day", "duration_min").show(5)

+--------------------+---------------------+-----------+-----------+-----------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|day_of_week|hour_of_day|     duration_min|
+--------------------+---------------------+-----------+-----------+-----------------+
| 2025-01-01 00:18:38|  2025-01-01 00:26:59|          4|          0|             8.35|
| 2025-01-01 00:32:40|  2025-01-01 00:35:13|          4|          0|             2.55|
| 2025-01-01 00:44:04|  2025-01-01 00:46:01|          4|          0|             1.95|
| 2025-01-01 00:14:27|  2025-01-01 00:20:01|          4|          0|5.566666666666666|
| 2025-01-01 00:21:34|  2025-01-01 00:25:06|          4|          0|3.533333333333333|
+--------------------+---------------------+-----------+-----------+-----------------+
only showing top 5 rows


In [4]:
demand = (with_cols
    .groupBy("pickup_borough", "day_of_week", "hour_of_day")    
    .agg(F.count("*").alias("trips"),                          
         F.avg("fare_amount").alias("avg_fare"),               
         F.avg("duration_min").alias("avg_duration_min"),      
         F.avg("trip_distance").alias("avg_distance_mi")))   

demand.filter("pickup_borough = 'Manhattan' AND day_of_week = 2").orderBy("hour_of_day").show(24)
demand.agg(F.sum("trips")).show()

+--------------+-----------+-----------+------+------------------+------------------+------------------+
|pickup_borough|day_of_week|hour_of_day| trips|          avg_fare|  avg_duration_min|   avg_distance_mi|
+--------------+-----------+-----------+------+------------------+------------------+------------------+
|     Manhattan|          2|          0| 26023|15.725083964185535|11.986787329157542|3.0421000653268275|
|     Manhattan|          2|          1| 13031|15.684398741462658|11.778309416007982|3.1141286163763318|
|     Manhattan|          2|          2|  7937|16.062461887362986| 11.63258577968166| 3.310099533828902|
|     Manhattan|          2|          3|  5891| 19.23536750976065|13.146627624059299| 4.130322525886946|
|     Manhattan|          2|          4|  7481|  24.0885469856971|14.325121418705166| 36.35404491378158|
|     Manhattan|          2|          5| 13791|20.015233848161845|12.755018973726827|12.896373721992612|
|     Manhattan|          2|          6| 32989| 16.2960

In [5]:
demand.write.mode("overwrite").partitionBy("pickup_borough").parquet("../output/m2_demand")

import os
print(sorted(os.listdir("../output/m2_demand")))

Py4JJavaError: An error occurred while calling o656.parquet.
: java.lang.RuntimeException: java.io.FileNotFoundException: java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset. -see https://cwiki.apache.org/confluence/display/HADOOP2/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:790)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:299)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:315)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:1179)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:861)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:901)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:873)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:900)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdir

In [6]:
import m2

demand2 = m2.build(clean, zones)
m2.write(demand2, "../output")
demand2.agg(F.sum("trips")).show()   

Py4JJavaError: An error occurred while calling o721.parquet.
: java.lang.RuntimeException: java.io.FileNotFoundException: java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset. -see https://cwiki.apache.org/confluence/display/HADOOP2/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:790)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:299)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:315)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:1179)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:861)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:901)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:873)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:900)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdir